# Depuración de texto y persistencia de datos con Python
**Actividad integradora de las unidades 3 y 4** · Programación I · Maestría en Ciencia de Datos (CUCEA)
Docente: Dr. Ramón Osvaldo Guardado Medina


| **Raúl Alfonso Cordero Villalobos**
| **MCD-B Octubre 2026** |

**Cómo reproducir:** coloca este notebook en la misma carpeta que `ejemplo_guiado.py`, `actividad_inicio.py` y la carpeta `datos/` (ZIP extraído completo). Requiere Python 3.10+ y solo biblioteca estándar. Ejecuta *Kernel → Restart & Run All*.

**Salidas que genera:** `salidas/` (reto), `salidas_demo/` (ejemplo guiado) y `pruebas/` (copias de prueba; `datos/` nunca se modifica).

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys

# En un notebook no existe __file__: la base es la carpeta del notebook.
BASE = Path.cwd().resolve()
DATOS = BASE / "datos"
ENTRADA = DATOS / "solicitudes.txt"
SALIDA = BASE / "salidas"

assert DATOS.is_dir(), "Abre el notebook en la carpeta que contiene datos/"
print("Python", sys.version.split()[0])
print("Archivos en datos/:", sorted(p.name for p in DATOS.iterdir()))

---
## Etapa 1 · Diagnóstico de cadenas (25 min)

### 1.1 Predicciones para `codigo = " mx-gdl-0042 "`

Posiciones: el espacio inicial ocupa el índice 0 y el final el índice 12.

| Expresión | Predicción | Razonamiento |
|---|---|---|
| `len(codigo)` | `13` | 11 caracteres visibles + 2 espacios |
| `codigo[2:4]` | `'x-'` | índices 2 y 3 (el 4 se excluye) |
| `codigo[-6:-2]` | `'-004'` | -6 equivale a 7 y -2 a 11 → índices 7 a 10 |
| `codigo.strip().upper()` | `'MX-GDL-0042'` | se quitan extremos y se pasa a mayúsculas |
| `codigo.find("xyz")` | `-1` | la subcadena no existe |

In [ ]:
codigo = " mx-gdl-0042 "
predicciones = {
    "len(codigo)": 13,
    "codigo[2:4]": "x-",
    "codigo[-6:-2]": "-004",
    'codigo.strip().upper()': "MX-GDL-0042",
    'codigo.find("xyz")': -1,
}
for expr, predicho in predicciones.items():
    observado = eval(expr)
    estado = "✓" if observado == predicho else "✗ DIFERENCIA"
    print(f"{expr:<25} predicho={predicho!r:<16} observado={observado!r:<16} {estado}")

In [ ]:
# ¿strip() modifica codigo si no se reasigna?
codigo.strip()
print(repr(codigo))                 # sigue con espacios: las cadenas son inmutables
codigo_limpio = codigo.strip()      # hay que reasignar a un nombre (nuevo o el mismo)
print(repr(codigo_limpio))

# in vs find()
print("gdl" in codigo, codigo.find("gdl"))   # bool  vs  posición
print("xyz" in codigo, codigo.find("xyz"))   # False vs  -1

# Por qué -1 no es una posición válida:
pos = codigo.find("xyz")
print(repr(codigo[pos]))   # ¡devuelve el último carácter! -1 es un índice negativo válido

**Respuestas**

- **¿`codigo.strip()` modifica `codigo` si no se reasigna?** No. Las cadenas en Python son inmutables: `strip()` devuelve una *nueva* cadena; si no se guarda, el resultado se pierde y `codigo` conserva sus espacios. Por eso el depurador guarda siempre el texto original y la versión transformada en variables distintas.
- **Diferencia entre `in` y `find()`:** `in` responde *si* la subcadena existe (`True`/`False`); `find()` responde *dónde* empieza (índice) o `-1` si no existe. Para decidir basta `in`; para extraer o recortar se necesita la posición.
- **¿Por qué `-1` no debe usarse como posición?** Porque `-1` es un índice negativo válido que apunta al último carácter. Si se usa sin verificar, `codigo[-1]` o `codigo[:-1]` producen resultados plausibles pero incorrectos, sin lanzar ningún error. Hay que comprobar `pos != -1` (o usar `index()`, que lanza `ValueError`).

### 1.2 Categorías en `etiquetas.txt`: antes y después de normalizar
Antes de limpiar solo se elimina el salto de línea (`rstrip("\n")`), para ver cuántas variantes de escritura existen realmente.

In [ ]:
def normalizar_categoria(texto: str) -> str:
    # Entrada: texto de categoría tal como viene en el archivo.
    # Salida: strip + lower, '-' y '_' → espacio, palabras unidas con '_'.
    # Conserva tildes (no se eliminan acentos).
    t = texto.strip().lower().replace("-", " ").replace("_", " ")
    return "_".join(t.split())   # split() sin argumento colapsa espacios repetidos

with (DATOS / "etiquetas.txt").open(encoding="utf-8") as f:
    crudas = [linea.rstrip("\n") for linea in f]

antes = set(crudas)
despues = {normalizar_categoria(e) for e in crudas}

for e in crudas:
    print(f"{e!r:<22} → {normalizar_categoria(e)!r}")
print(f"\nCategorías distintas antes: {len(antes)}  ·  después: {len(despues)}")
print("Después:", sorted(despues))

In [ ]:
# Caso donde capitalize() altera una sigla que debe conservarse
for texto in ["SPEI rechazado", "acceso al SIIAU", "error en API REST"]:
    print(f"{texto!r:<22} → {texto.capitalize()!r}")

`capitalize()` pone en mayúscula **solo** el primer carácter y convierte **todo lo demás a minúsculas**: `"SPEI rechazado"` se vuelve `"Spei rechazado"` y `"acceso al SIIAU"` pierde la sigla (`"Acceso al siiau"`). Por eso la regla de `descripcion` del contrato **no** cambia mayúsculas: se pierde información (siglas, nombres propios) que no se puede recuperar después.

---
## Etapa 2 · Ejemplo guiado
Se ejecuta `ejemplo_guiado.py` **dos veces** como proceso independiente (igual que en terminal) y se inspeccionan sus salidas.

In [ ]:
demo = BASE / "salidas_demo"
for corrida in (1, 2):
    r = subprocess.run([sys.executable, "ejemplo_guiado.py"], cwd=BASE,
                       capture_output=True, text=True, encoding="utf-8")
    print(f"--- Ejecución {corrida} ---\n{r.stdout}{r.stderr}")
    with (demo / "errores.txt").open(encoding="utf-8") as f:
        n_err = sum(1 for _ in f)
    with (demo / "ejecuciones.log").open(encoding="utf-8") as f:
        n_log = sum(1 for _ in f)
    print(f"errores.txt: {n_err} líneas · ejecuciones.log: {n_log} líneas\n")

**¿Por qué `errores.txt` conserva dos líneas y `ejecuciones.log` aumenta?**

| Modo | Uso en el ejemplo | Efecto |
|---|---|---|
| `r` (por defecto) | `eventos.txt` | solo lectura; falla con `FileNotFoundError` si no existe |
| `w` | `errores.txt` | **trunca** el archivo al abrirlo: cada ejecución lo reescribe desde cero → siempre 2 líneas |
| `a` | `ejecuciones.log` | **agrega** al final sin borrar: cada ejecución suma una línea → historial |

**Dónde aparece cada operación:**

- `strip()` y `split(":")` → `etiqueta.strip().split(":")` separa `"sensor_norte"` y `"23.5"` (desempaquetado en `nombre, valor`).
- `replace()` y `capitalize()` → `nombre.replace("_", " ").capitalize()` produce `Sensor norte`.
- f-string → `f"Valor: {float(valor):.1f}"` y `f"Errores detectados: {total}\n"`.
- Cierre automático → los bloques `with ... as entrada/salida/historial` cierran el archivo al salir del bloque, aunque ocurra una excepción.
- Además: `" ERROR " in linea` usa `in` (con espacios para no coincidir con `ERRORES`), y el `try/except FileNotFoundError` reporta `error.filename`.

---
## Etapa 3 · Implementación del depurador (65 min)

### Contrato de datos
| Campo | Transformación | Regla de aceptación |
|---|---|---|
| `id` | `strip().upper()` | `SOL-` + 4 dígitos ASCII, longitud 8, único entre aceptados |
| `correo` | `strip().lower()` | una sola `@`, partes no vacías, sin espacios, punto en el dominio que no sea primero ni último |
| `categoria` | `normalizar_categoria()` | `acceso`, `pago_en_línea` o `soporte_técnico` |
| `descripcion` | strip + espacios internos a uno | no vacía; conserva mayúsculas y tildes |

**Orden obligatorio (se registra solo el primer motivo):** línea vacía → campos ≠ 4 → campo vacío → ID inválido → correo inválido → categoría fuera de catálogo → ID duplicado. El ID entra a `ids_vistos` **solo** si el registro completo se acepta.

In [ ]:
ENCABEZADO = "id|correo|categoria|descripcion"
CATALOGO = {"acceso", "pago_en_línea", "soporte_técnico"}
DIGITOS_ASCII = set("0123456789")   # isdigit() aceptaría '٣' o '²'; aquí solo 0–9

MOTIVOS = {
    "vacia": "línea vacía",
    "num_campos": "cantidad de campos distinta de cuatro",
    "campo_vacio": "campo vacío",
    "id": "ID con formato inválido",
    "correo": "correo inválido",
    "categoria": "categoría fuera del catálogo",
    "duplicado": "ID duplicado",
}


class EncabezadoInvalidoError(ValueError):
    # El archivo de entrada no respeta el contrato de cabecera.
    pass


def id_valido(id_: str) -> bool:
    # Entrada: ID ya normalizado. Salida: True si es 'SOL-' + 4 dígitos ASCII.
    return (len(id_) == 8 and id_.startswith("SOL-")
            and all(c in DIGITOS_ASCII for c in id_[4:]))


def correo_valido(correo: str) -> bool:
    # Validación didáctica (no RFC 5322): una @, partes no vacías,
    # sin espacios y un punto en el dominio que no sea su primer ni último carácter.
    if correo.count("@") != 1 or any(c.isspace() for c in correo):
        return False
    usuario, dominio = correo.split("@")
    if not usuario or not dominio:
        return False
    return "." in dominio and not dominio.startswith(".") and not dominio.endswith(".")


def normalizar_descripcion(texto: str) -> str:
    # Quita espacios extremos y reduce espacios internos consecutivos a uno.
    return " ".join(texto.split())


def validar_campos(campos: list[str], ids_vistos: set[str]):
    # Entrada: lista de campos crudos (resultado de split('|')) y conjunto de IDs aceptados.
    # Salida: (registro_limpio, None) si es válido; (None, motivo) si se rechaza.
    # No modifica ids_vistos: eso lo hace el flujo principal al aceptar.
    if len(campos) != 4:
        return None, MOTIVOS["num_campos"]
    if any(c.strip() == "" for c in campos):
        return None, MOTIVOS["campo_vacio"]

    id_ = campos[0].strip().upper()
    correo = campos[1].strip().lower()
    categoria = normalizar_categoria(campos[2])
    descripcion = normalizar_descripcion(campos[3])

    if not id_valido(id_):
        return None, MOTIVOS["id"]
    if not correo_valido(correo):
        return None, MOTIVOS["correo"]
    if categoria not in CATALOGO:
        return None, MOTIVOS["categoria"]
    if id_ in ids_vistos:
        return None, MOTIVOS["duplicado"]

    registro = {"id": id_, "correo": correo,
                "categoria": categoria, "descripcion": descripcion}
    return registro, None

### Pruebas unitarias rápidas de las funciones

In [ ]:
assert normalizar_categoria("  PAGO_EN_LÍNEA ") == "pago_en_línea"
assert normalizar_categoria("soporte-técnico") == "soporte_técnico"
assert id_valido("SOL-0001") and not id_valido("SOL-00A1") and not id_valido("XX-0026")
assert not id_valido("SOL-٠٠١٢")          # dígitos arábigos: isdigit() los aceptaría
assert correo_valido("a@b.com")
for malo in ["a@@b.com", "sin_arroba.example.com", "a@.com", "a@com.", "a b@c.com", "@b.com", "a@b"]:
    assert not correo_valido(malo), malo
assert normalizar_descripcion("  Hola    mundo  ") == "Hola mundo"
assert validar_campos(["SOL-0001", "x@y.mx", "Acceso"], set()) == (None, MOTIVOS["num_campos"])
assert validar_campos([" sol-0001 ", "X@Y.MX", "ACCESO", " ok "], {"SOL-0001"}) == (None, MOTIVOS["duplicado"])
print("Todas las pruebas unitarias pasaron ✓")

### Flujo principal
`procesar()` recibe la ruta de entrada y la carpeta de salida como parámetros (así las pruebas usan copias sin tocar `datos/`). Decisiones de diseño:

- La cabecera se verifica **antes** de abrir cualquier salida en modo `w`, para no borrar resultados anteriores si la entrada es incorrecta.
- La entrada se recorre línea por línea (`for ... in enumerate(entrada, start=2)`), nunca con `read()` ni `readlines()`.
- Cada línea de datos produce exactamente un registro válido o un rechazo.
- El texto original del rechazo se guarda sin el salto final (`rstrip("\r\n")`), sin ninguna otra limpieza.
- `FileNotFoundError` se captura en `main()`; los errores de programación **no** se capturan.

In [ ]:
def procesar(entrada: Path, salida: Path) -> dict:
    # Depura `entrada` y escribe en `salida`: solicitudes_limpias.txt, rechazos.jsonl y resumen.json (modo w).
    # Devuelve los contadores calculados en memoria.
    # Lanza FileNotFoundError si no existe la entrada y EncabezadoInvalidoError si la cabecera no coincide.
    with entrada.open("r", encoding="utf-8") as f_in:
        encabezado = f_in.readline().rstrip("\r\n")     # lee SOLO la primera línea
        if encabezado != ENCABEZADO:
            raise EncabezadoInvalidoError(
                f"Encabezado inesperado en {entrada.name}: {encabezado!r} "
                f"(se esperaba {ENCABEZADO!r}). No se modificaron las salidas.")

        salida.mkdir(parents=True, exist_ok=True)
        total = validos = rechazados = vacias = 0
        por_categoria: dict[str, int] = {}
        ids_vistos: set[str] = set()
        lineas_rechazadas: list[int] = []

        with (salida / "solicitudes_limpias.txt").open("w", encoding="utf-8") as f_ok, \
             (salida / "rechazos.jsonl").open("w", encoding="utf-8") as f_rej:
            f_ok.write(ENCABEZADO + "\n")

            for num_linea, linea in enumerate(f_in, start=2):
                total += 1
                original = linea.rstrip("\r\n")

                if original.strip() == "":
                    vacias += 1
                    registro, motivo = None, MOTIVOS["vacia"]
                else:
                    registro, motivo = validar_campos(original.split("|"), ids_vistos)

                if registro is None:
                    rechazados += 1
                    lineas_rechazadas.append(num_linea)
                    f_rej.write(json.dumps({"linea": num_linea, "motivo": motivo,
                                            "texto_original": original},
                                           ensure_ascii=False) + "\n")
                else:
                    validos += 1
                    ids_vistos.add(registro["id"])           # solo al aceptar
                    cat = registro["categoria"]
                    por_categoria[cat] = por_categoria.get(cat, 0) + 1
                    f_ok.write("|".join(registro.values()) + "\n")

    resumen = {
        "total": total,
        "validos": validos,
        "rechazados": rechazados,
        "por_categoria": dict(sorted(por_categoria.items())),
        "lineas_vacias": vacias,
        "lineas_rechazadas": lineas_rechazadas,
        "tasa_aceptacion": round(validos / total, 4) if total else 0.0,
        "tasa_rechazo": round(rechazados / total, 4) if total else 0.0,
    }
    with (salida / "resumen.json").open("w", encoding="utf-8") as f_res:
        json.dump(resumen, f_res, ensure_ascii=False, indent=2)
    return resumen

---
## Etapa 4 · Persistencia y verificación (40 min)

| Salida | Modo | Contenido |
|---|---|---|
| `solicitudes_limpias.txt` | `w` | cabecera original + válidos en orden de entrada, separados por `\|` |
| `rechazos.jsonl` | `w` | un objeto JSON por línea: `linea`, `motivo`, `texto_original` |
| `resumen.json` | `w` | `total`, `validos`, `rechazados`, `por_categoria` (+ tasas y líneas) |
| `ejecuciones.log` | `a` | una línea por ejecución **exitosa** (fecha y hora, entrada, conteos) |

`verificar_salidas()` **reabre** los archivos persistidos y compara contra los contadores; no basta con imprimir.

In [ ]:
def verificar_salidas(salida: Path, esperado: dict) -> None:
    # Reabre los archivos persistidos y comprueba que coincidan con los contadores en memoria.
    # Lanza AssertionError con un mensaje claro si algo no cuadra.
    with (salida / "solicitudes_limpias.txt").open(encoding="utf-8") as f:
        cabecera = f.readline().rstrip("\n")
        limpios = sum(1 for _ in f)              # excluye la cabecera
    with (salida / "resumen.json").open(encoding="utf-8") as f:
        resumen = json.load(f)
    rechazos = []
    with (salida / "rechazos.jsonl").open(encoding="utf-8") as f:
        for n, linea in enumerate(f, start=1):
            obj = json.loads(linea)              # cada línea debe ser JSON válido
            assert set(obj) == {"linea", "motivo", "texto_original"}, f"rechazos.jsonl línea {n}"
            rechazos.append(obj)

    assert cabecera == ENCABEZADO, "cabecera de solicitudes_limpias.txt"
    assert limpios == esperado["validos"] == resumen["validos"], "conteo de válidos"
    assert len(rechazos) == esperado["rechazados"] == resumen["rechazados"], "conteo de rechazados"
    assert resumen["total"] == resumen["validos"] + resumen["rechazados"], "total = validos + rechazados"
    assert sum(resumen["por_categoria"].values()) == resumen["validos"], "suma por_categoria = validos"
    assert [r["linea"] for r in rechazos] == resumen["lineas_rechazadas"], "líneas rechazadas"


def registrar_ejecucion(salida: Path, entrada: Path, resumen: dict) -> None:
    # Agrega (modo 'a') una línea al historial; solo se llama tras una ejecución verificada.
    marca = datetime.now().isoformat(timespec="seconds")
    with (salida / "ejecuciones.log").open("a", encoding="utf-8") as f:
        f.write(f"{marca} | entrada={entrada.name} | total={resumen['total']} "
                f"| validos={resumen['validos']} | rechazados={resumen['rechazados']}\n")


def main(entrada: Path = ENTRADA, salida: Path = SALIDA):
    # Orquesta el proceso y traduce errores esperados en mensajes claros.
    try:
        resumen = procesar(entrada, salida)
        verificar_salidas(salida, resumen)
        registrar_ejecucion(salida, entrada, resumen)
    except FileNotFoundError as error:
        print(f"No se encontró el archivo de entrada: {error.filename}")
        return None
    except EncabezadoInvalidoError as error:
        print(f"Proceso detenido: {error}")
        return None
    print(f"OK · {entrada.name}: total={resumen['total']} · válidos={resumen['validos']} "
          f"· rechazados={resumen['rechazados']}")
    return resumen

### Ejecución sobre los datos originales

In [ ]:
resumen = main()
print(json.dumps(resumen, ensure_ascii=False, indent=2))

In [ ]:
print("— rechazos.jsonl —")
with (SALIDA / "rechazos.jsonl").open(encoding="utf-8") as f:
    for linea in f:
        r = json.loads(linea)
        print(f"línea {r['linea']:>2} | {r['motivo']:<38} | {r['texto_original']!r}")

print("\n— solicitudes_limpias.txt (primeras 5 líneas) —")
with (SALIDA / "solicitudes_limpias.txt").open(encoding="utf-8") as f:
    for _, linea in zip(range(5), f):
        print(linea, end="")

### Controles esperados vs. observados
Los valores esperados vienen del documento de la actividad; los observados se **leen de los archivos persistidos**, no de constantes.

In [ ]:
with (SALIDA / "resumen.json").open(encoding="utf-8") as f:
    obs = json.load(f)

controles = [
    ("Líneas de datos (incluida la vacía)", 30, obs["total"]),
    ("Registros válidos", 24, obs["validos"]),
    ("Registros rechazados", 6, obs["rechazados"]),
    ("acceso", 9, obs["por_categoria"].get("acceso")),
    ("pago_en_línea", 9, obs["por_categoria"].get("pago_en_línea")),
    ("soporte_técnico", 6, obs["por_categoria"].get("soporte_técnico")),
    ("Categorías en etiquetas.txt antes", 9, len(antes)),
    ("Categorías en etiquetas.txt después", 3, len(despues)),
    ("Líneas físicas rechazadas", [26, 27, 28, 29, 30, 31], obs["lineas_rechazadas"]),
    ("Tasa de rechazo", "20 %", f"{obs['tasa_rechazo']:.0%}".replace("%", " %")),
    ("Tasa de aceptación", "80 %", f"{obs['tasa_aceptacion']:.0%}".replace("%", " %")),
    ("total = validos + rechazados", True, obs["total"] == obs["validos"] + obs["rechazados"]),
    ("suma(por_categoria) = validos", True, sum(obs["por_categoria"].values()) == obs["validos"]),
]
for nombre, esp, ob in controles:
    print(f"{'✓' if esp == ob else '✗'} {nombre:<38} esperado={esp!s:<22} observado={ob}")
assert all(esp == ob for _, esp, ob in controles)

### Pruebas mínimas con evidencia
Todas las pruebas que requieren entradas modificadas trabajan sobre **copias** en `pruebas/`; `datos/` no se altera (se comprueba al final con una huella SHA-256).

In [ ]:
import hashlib, shutil

def huella(ruta: Path) -> str:
    h = hashlib.sha256()
    with ruta.open("rb") as f:
        for bloque in iter(lambda: f.read(8192), b""):
            h.update(bloque)
    return h.hexdigest()

def contar_lineas(ruta: Path) -> int:
    with ruta.open(encoding="utf-8") as f:
        return sum(1 for _ in f)

huella_original = huella(ENTRADA)
PRUEBAS = BASE / "pruebas"
if PRUEBAS.exists():
    shutil.rmtree(PRUEBAS)          # las pruebas parten de cero en cada ejecución
PRUEBAS.mkdir()
evidencia = []                      # (prueba, esperado, observado, ok)

**Prueba 1 · Dos ejecuciones:** las salidas de datos conservan sus conteos y el historial crece exactamente una línea.

In [ ]:
log = SALIDA / "ejecuciones.log"
r1 = main(); log_1 = contar_lineas(log); limpias_1 = contar_lineas(SALIDA / "solicitudes_limpias.txt")
r2 = main(); log_2 = contar_lineas(log); limpias_2 = contar_lineas(SALIDA / "solicitudes_limpias.txt")

ok = (r1 == r2 and limpias_1 == limpias_2 and log_2 == log_1 + 1)
evidencia.append(("1. Doble ejecución", "mismos conteos; log +1 línea",
                  f"limpias {limpias_1}→{limpias_2}; log {log_1}→{log_2}", ok))
print(evidencia[-1])

**Prueba 2 · Ruta inexistente:** se informa qué archivo falta, sin traceback.

In [ ]:
ruta_falsa = DATOS / "no_existe.txt"
salida_p2 = PRUEBAS / "salidas_ruta_inexistente"
res = main(ruta_falsa, salida_p2)
ok = res is None and not salida_p2.exists()
evidencia.append(("2. Ruta inexistente", "mensaje con nombre del archivo; sin salidas",
                  f"retorno={res}; carpeta creada={salida_p2.exists()}", ok))
print(evidencia[-1])

**Prueba 3 · Copia con `SOL-00A1` y `a@@b.com`** en registros distintos (se agregan al final de una copia).

In [ ]:
copia = PRUEBAS / "solicitudes_prueba3.txt"
shutil.copyfile(ENTRADA, copia)
with copia.open("a", encoding="utf-8") as f:
    f.write("SOL-00A1|alumno40@cucea.udg.mx|Acceso|ID con letra\n")      # línea 32
    f.write("SOL-0041|a@@b.com|Acceso|Correo con doble arroba\n")         # línea 33

salida_p3 = PRUEBAS / "salidas_prueba3"
res3 = main(copia, salida_p3)
with (salida_p3 / "rechazos.jsonl").open(encoding="utf-8") as f:
    nuevos = {r["linea"]: r["motivo"] for r in map(json.loads, f) if r["linea"] >= 32}

esperado3 = {32: MOTIVOS["id"], 33: MOTIVOS["correo"]}
ok = nuevos == esperado3 and res3["rechazados"] == 8 and res3["validos"] == 24
evidencia.append(("3. SOL-00A1 y a@@b.com", f"{esperado3}; 24 válidos / 8 rechazados",
                  f"{nuevos}; {res3['validos']} válidos / {res3['rechazados']} rechazados", ok))
print(evidencia[-1])

**Prueba 4 · Cabecera incorrecta:** el proceso se detiene antes de reemplazar resultados anteriores.

In [ ]:
copia4 = PRUEBAS / "solicitudes_cabecera_mala.txt"
with ENTRADA.open(encoding="utf-8") as f_in, copia4.open("w", encoding="utf-8") as f_out:
    f_in.readline()
    f_out.write("ID|Correo|Categoria|Descripcion\n")      # cabecera alterada
    for linea in f_in:
        f_out.write(linea)

# Se apunta a la carpeta de salidas real para demostrar que NO se toca.
antes_p4 = {p.name: huella(p) for p in SALIDA.iterdir()}
res4 = main(copia4, SALIDA)
despues_p4 = {p.name: huella(p) for p in SALIDA.iterdir()}

ok = res4 is None and antes_p4 == despues_p4
evidencia.append(("4. Cabecera incorrecta", "se detiene; salidas previas intactas",
                  f"retorno={res4}; salidas sin cambios={antes_p4 == despues_p4}", ok))
print(evidencia[-1])

In [ ]:
evidencia.append(("datos/ intacto", "misma huella SHA-256", f"igual={huella(ENTRADA) == huella_original}",
                  huella(ENTRADA) == huella_original))

print(f"{'Prueba':<26} | {'Esperado':<44} | {'Observado':<50} | OK")
print("-" * 130)
for prueba, esp, obs_, ok in evidencia:
    print(f"{prueba:<26} | {str(esp):<44} | {str(obs_):<50} | {'✓' if ok else '✗'}")
assert all(ok for *_, ok in evidencia)

### Extensión opcional · Memoria de `read()` vs iteración por líneas
Se genera un archivo grande en `pruebas/` y se mide el pico de memoria con `tracemalloc`.

In [ ]:
import tracemalloc

grande = PRUEBAS / "solicitudes_grande.txt"
N = 200_000
with grande.open("w", encoding="utf-8") as f:
    f.write(ENCABEZADO + "\n")
    for i in range(N):
        f.write(f"SOL-{i % 10000:04d}|alumno{i}@cucea.udg.mx|Acceso|Descripción de prueba número {i}\n")

def pico(funcion):
    tracemalloc.start()
    funcion()
    _, maximo = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return maximo / 1_048_576

def con_read():
    with grande.open(encoding="utf-8") as f:
        return f.read().count("\n")

def por_lineas():
    with grande.open(encoding="utf-8") as f:
        return sum(1 for _ in f)

def ids_unicos():
    vistos = set()
    with grande.open(encoding="utf-8") as f:
        next(f)
        for i, linea in enumerate(f):
            vistos.add(f"{linea[:8]}-{i}")   # simula N IDs distintos
    return len(vistos)

print(f"Tamaño en disco: {grande.stat().st_size / 1_048_576:.1f} MB")
print(f"Pico con read():           {pico(con_read):8.2f} MB")
print(f"Pico iterando por líneas:  {pico(por_lineas):8.2f} MB")
print(f"Pico con set de {N:,} IDs: {pico(ids_unicos):8.2f} MB")
grande.unlink()   # no incluir el archivo grande en el ZIP de entrega

La iteración por líneas mantiene la memoria casi constante (una línea y el búfer del archivo), mientras que `read()` crece con el tamaño del archivo. Sin embargo, el depurador **no** es de memoria constante: el conjunto `ids_vistos` crece con cada ID único aceptado. Para archivos que superen la memoria, el control de duplicados tendría que moverse a un almacén externo (p. ej. `sqlite3` con índice único, o particionar por prefijo de ID y procesar por lotes).

---
## Etapa 5 · Interpretación y cierre (30 min)
Datos de apoyo calculados para el informe (2–3 páginas). El texto de abajo es un **borrador** para desarrollar en el informe.

In [ ]:
# Distribución SIN normalizar: solo strip() de la categoría, en líneas con 4 campos
sin_normalizar: dict[str, int] = {}
with ENTRADA.open(encoding="utf-8") as f:
    next(f)
    for linea in f:
        campos = linea.rstrip("\r\n").split("|")
        if len(campos) == 4 and campos[2].strip():
            cat = campos[2].strip()
            sin_normalizar[cat] = sin_normalizar.get(cat, 0) + 1

print("Sin normalizar (solo strip):")
for cat, n in sorted(sin_normalizar.items(), key=lambda kv: -kv[1]):
    print(f"  {cat!r:<20} {n}")
print(f"→ {len(sin_normalizar)} categorías aparentes")
print("\nNormalizada (registros válidos):", obs["por_categoria"])

**1. ¿Cómo cambia la distribución sin normalizar?** Sin normalizar aparecen 9 "categorías" aparentes en lugar de 3, cada una con entre 2 y 6 solicitudes; el conteo de cualquier categoría real queda fragmentado y ninguna variante refleja la demanda real (p. ej., `acceso` aparece repartido entre `Acceso`, `ACCESO` y `acceso`). Normalizado: acceso 9, pago_en_línea 9, soporte_técnico 6.

**2. ¿Qué sesgo introduce eliminar todos los rechazos?** Los 6 rechazos (20 %) no son aleatorios: 4 de ellos traen `Acceso` como categoría y otro la tiene vacía. Si se descartan, `acceso` queda subrepresentada respecto a la demanda real. Además, los rechazos suelen venir de canales o usuarios específicos (captura manual, formularios viejos), por lo que eliminarlos puede ocultar a un subgrupo. Por eso se conservan en `rechazos.jsonl` con línea, motivo y texto original.

**3. ¿Por qué la tasa de aceptación (80 %) no demuestra exactitud?** Las reglas verifican *forma*, no *verdad*: un correo bien formado puede no existir, una categoría válida puede estar mal asignada, y la validación de correo es didáctica. Además, solo se registra el primer motivo, así que una línea puede tener más problemas de los que el conteo indica.

**4. ¿Qué información se pierde con cada transformación?** `upper()/lower()` borra la grafía original (no se sabe quién escribió `ACCESO`); la normalización de categoría elimina la distinción entre `-`, `_` y espacio (útil para rastrear el origen del dato); colapsar espacios en la descripción pierde formato intencional; un ID duplicado rechazado pierde una solicitud que podría ser legítima (SOL-0001 de otro correo). La trazabilidad se preserva gracias al archivo de rechazos con el texto original.

**5. ¿Qué cambiaría si una descripción contuviera `|` o si el archivo superara la memoria?** Con `|` dentro de un campo, `split("|")` produciría más de 4 campos y rechazaría registros válidos; habría que usar un formato con comillas y `csv.reader(f, delimiter="|")`, o `split("|", maxsplit=3)` si solo la última columna puede contenerlo. Si el archivo supera la memoria, la lectura ya es incremental, pero `ids_vistos` crece con los IDs únicos (ver extensión); se resolvería con `sqlite3`, particionado o un filtro de Bloom con verificación posterior.

### Uso de IA (declaración requerida)
_Completar:_ indicar qué partes apoyó la IA (p. ej., estructura del notebook y primera versión de las funciones) y cómo se verificó el código: pruebas unitarias de funciones, controles esperados vs. observados leídos de los archivos persistidos, las cuatro pruebas mínimas y la huella SHA-256 de `datos/`.